In [ ]:
# TODO 
# Code cleanup:
# - Remove commented-out code and unused imports.
# - Add docstrings to functions and classes for better readability.
# - Use consistent variable naming conventions.
# - Remove hardcoded file paths and replace them with configurable parameters.
# add function to safe guard order of points in homography
# try to remove manual boundary lineant one
# improve sift or add sift here. 

In [ ]:
import cv2
import numpy as np

# Load template image (clear view of socket)
template = cv2.imread("Data/frame_000449.png")
if template is None:
    print("Error: Could not load template image.")
    exit(1)
clone = template.copy()
points = []

def click_event(event, x, y, flags, param):
    global points, template

    if event == cv2.EVENT_LBUTTONDOWN:
        points.append([x, y])
        cv2.circle(template, (x, y), 5, (0,255,0), -1)
        cv2.imshow("Select 4 Corners", template)

cv2.imshow("Select 4 Corners", template)
cv2.setMouseCallback("Select 4 Corners", click_event)

print("Click 4 corners of the socket (clockwise)")
cv2.waitKey(0)
cv2.destroyAllWindows()

points = np.array(points, dtype=np.float32)

def order_points(pts):
    pts = pts.reshape(4, 2)

    rect = np.zeros((4, 2), dtype="float32")

    # sum → top-left (smallest), bottom-right (largest)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # top-left
    rect[2] = pts[np.argmax(s)]  # bottom-right

    # diff → top-right (smallest), bottom-left (largest)
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # top-right
    rect[3] = pts[np.argmax(diff)]  # bottom-left

    return rect.reshape(-1,1,2)

points = order_points(points)

# Save corners for reuse
np.save("template_corners.npy", points)

print("Saved template corners:", points)

In [ ]:
#---------Code for annotation of boundaries for homography---------

import cv2
import json
import os

# ---------- CONFIG ----------
DATA_PATH = "Data"
IMAGE_IDS = [319, 333, 353, 359, 365, 400,426, 449, 461, 468, 471, 496, 531]   # add all your frames here
OUTPUT_JSON = "manual_power_socket_boundaries.json"

# ---------- STORAGE ----------
all_boundaries = {}

current_points = []
current_image = None
current_frame_id = None

# ---------- MOUSE CALLBACK ----------
def click_event(event, x, y, flags, param):
    global current_points, current_image

    if event == cv2.EVENT_LBUTTONDOWN:
        if len(current_points) < 4:
            current_points.append([x, y])

            # draw point
            cv2.circle(current_image, (x, y), 5, (0,255,0), -1)

            # draw line if more than 1 point
            if len(current_points) > 1:
                cv2.line(
                    current_image,
                    tuple(current_points[-2]),
                    tuple(current_points[-1]),
                    (255,0,0), 2
                )

            cv2.imshow("Annotate", current_image)


# ---------- MAIN LOOP ----------
for frame_id in IMAGE_IDS:

    print(f"\nAnnotating frame: {frame_id}")

    img_path = os.path.join(DATA_PATH, f"frame_000{frame_id}.png")
    img = cv2.imread(img_path)

    if img is None:
        print(f"Image not found: {img_path}")
        continue

    current_image = img.copy()
    current_points = []
    current_frame_id = frame_id

    cv2.imshow("Annotate", current_image)
    cv2.setMouseCallback("Annotate", click_event)

    print("Click 4 corners (clockwise or counter-clockwise)")
    print("Press 'r' to reset, 'n' to save & go next, 'q' to quit")

    while True:
        key = cv2.waitKey(1) & 0xFF

        # Reset
        if key == ord('r'):
            current_image = img.copy()
            current_points = []
            cv2.imshow("Annotate", current_image)

        # Next image (save)
        elif key == ord('n'):
            if len(current_points) != 4:
                print("Need exactly 4 points!")
                continue
                
            all_boundaries[frame_id] = current_points.copy()
            print(f"Saved {frame_id}: {current_points}")
            break

        # Quit early
        elif key == ord('q'):
            print("Exiting early...")
            break

    if key == ord('q'):
        break

cv2.destroyAllWindows()

# ---------- SAVE JSON ----------
with open(OUTPUT_JSON, "w") as f:
    json.dump(all_boundaries, f, indent=4)

print(f"\nSaved all annotations to {OUTPUT_JSON}")

In [52]:
# loading the intrinsics

import numpy as np
import json

def load_intrinsics(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)

    K = np.array(data["camera_matrix"], dtype=np.float64)

    # Optional extras (useful later)
    width = data.get("image_width", None)
    height = data.get("image_height", None)
    dist = np.array(data.get("distortion_coefficients", []), dtype=np.float64)

    return K, width, height, dist

K, width, height, dist = load_intrinsics("Data/intrinsic.json")

print("K:\n", K)

K:
 [[1.47700975e+03 0.00000000e+00 1.29825015e+03]
 [0.00000000e+00 1.48044245e+03 6.86820162e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]


In [53]:
# class for managing tracks across frames

class TrackManager:
    def __init__(self, kp_ref, ref_frame_id):
        self.tracks = {}

        # initialize tracks from keyframe
        for i, kp in enumerate(kp_ref):
            self.tracks[i] = [(ref_frame_id, kp.pt)]

    def add_matches(self, matches, kp_curr, curr_frame_id):
        for m in matches:
            ref_id = m.queryIdx
            pt = kp_curr[m.trainIdx].pt

            self.tracks[ref_id].append((curr_frame_id, pt))

    def get_valid_tracks(self, min_length=3):
        return {
            k: v for k, v in self.tracks.items()
            if len(v) >= min_length
        }

In [54]:
def sift_in_bbox(img, bbox):
    x, y, w, h = bbox

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    mask = np.zeros(gray.shape, dtype=np.uint8)
    mask[y:y+h, x:x+w] = 255

    sift = cv2.SIFT_create ()
    kp, des = sift.detectAndCompute(gray, mask)
    # kp, des = sift.detectAndCompute(gray, mask)
    # return kp, des
    
    # Convert to RootSIFT
    def rootsift(descriptors):
    # L1 normalize
        descriptors /= (descriptors.sum(axis=1, keepdims=True) + 1e-7)
    
        # Square root
        descriptors = np.sqrt(descriptors)
    
        return descriptors

    des_root = rootsift(des)

    return kp, des_root 

def match_and_visualise(frame_id, img2, kp1, des1, kp2, des2, image_ref_boundary_pts):
    # Step 2: FLANN matcher
    # FLANN_INDEX_KDTREE = 1
# index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
# search_params = dict(checks=50)

# flann = cv2.FlannBasedMatcher(index_params, search_params)
# matches = flann.knnMatch(des1, des2, k=2)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    # Step 3: Lowe ratio test
    good_matches = []
    for m, n in matches:
        if m.distance < 0.6 * n.distance: #0.75 is a common choice, can be tuned
            good_matches.append(m)

    print("Good matches:", len(good_matches))

    if len(good_matches) >= 4:
        pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)
        pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)

        # Step 4: Homography
        H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 6.5) #5.0

        print("Inliers:", np.sum(mask)) # can add inliers to output if needed

        if H is None:
            print("Homography failed:\n", H)
            return None
        
        inliers = mask.ravel().astype(bool)
        filtered_matches = [m for i, m in enumerate(good_matches) if inliers[i]]
        
        # Step 5: Project boundary
        projected = cv2.perspectiveTransform(image_ref_boundary_pts, H)
        print(projected)
        
        output = img2.copy()
        cv2.polylines(output, [np.int32(projected)], True, (0,255,0), 3)
        cv2.imshow("Boundary Detection frame " + str(frame_id), output)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
        
        return projected, filtered_matches
    else:
        print("Not enough matches for homography")
        return None, None

In [56]:
import cv2
import numpy as np
import json
import os

# -------- CONFIG --------
image_dir = "Data"
image_prefix = "frame_000"
image_ext = ".png"
file_name_prefix = "bboxes_"
file_name_suffix = ".json"

# device_type = "ethernet_socket"  
device_type = "power_socket"  
# device_type = "vga_socket" 

file_name = file_name_prefix + device_type + file_name_suffix
bbox_file = os.path.join(image_dir, file_name)
valid_image_ids = [353, 359, 365, 400, 426, 449, 461, 471, 496, 531]   #power socket

def get_image_path(i):
    path = image_dir + "/" + image_prefix + str(i) + image_ext
    return path

keyframe_idx = 426
image_path = get_image_path(keyframe_idx)

# ---------- LOAD DATA ----------

# boundary data -----------------
# boundary_filename = "manual_power_socket_boundaries.json"
# boundary_filename = "manual_ethernet_socket_boundaries.json"
boundary_filename = "manual_power_socket_boundaries.json"



with open(boundary_filename, "r") as f:
    boundaries = json.load(f)
    
# bounding box data -----------------
with open(bbox_file, "r") as f:
    bboxes = json.load(f)
    
frame_ids = list(boundaries.keys()) # replace with image indices later
# convert frame ids to integers for easier handling
frame_ids = [int(fid) for fid in frame_ids if int(fid) in valid_image_ids]
print(frame_ids)

#------------------------------------

# ---------- LOAD KEYFRAME ----------
image_ref = cv2.imread(image_path)
bbox_ref = bboxes.get(str(keyframe_idx))
if bbox_ref is None:
    print(f"Error: No bounding box found for keyframe {keyframe_idx}")
    exit(1)
    
kp_ref, des_ref = sift_in_bbox(image_ref, bbox_ref)
image_ref_boundary_pts = np.array(boundaries[str(keyframe_idx)], dtype=np.float32).reshape(-1,1,2)
#------------------------------------

# ---------- Initialize Track Manager ----------
track_manager = TrackManager(kp_ref, int(keyframe_idx))
#------------------------------------

# ---------- STORE ALL PROJECTED POINTS ----------
all_projected_boundary_pts = {}

# include keyframe itself
all_projected_boundary_pts[keyframe_idx] = image_ref_boundary_pts.reshape(4,2)

# ---------- LOOP THROUGH FRAMES ----------
for frame_id in frame_ids:
    
    if frame_id == keyframe_idx:
        continue

    print(f"\nProcessing frame {frame_id}")

    img2 = cv2.imread(get_image_path(frame_id))
    bbox2 = bboxes.get(str(frame_id))
    kp2, des2 = sift_in_bbox(img2, bbox2)

    if img2 is None or bbox2 is None:
        print("Missing data")
        continue
    
    if des2 is None:
        print("Descriptors missing, skipping")
        continue
    
    print(f"Keypoints: {len(kp_ref)} vs {len(kp2)}")
    
    all_projected_boundary_pts[frame_id], filtered_matches = match_and_visualise(frame_id, img2, kp_ref, des_ref, kp2, des2, image_ref_boundary_pts) 
    if filtered_matches is None:
        print("No valid matches, skipping track update")
        continue
    
    if all_projected_boundary_pts[frame_id] is not None:
        all_projected_boundary_pts[frame_id] = all_projected_boundary_pts[frame_id].reshape(4,2)
    

    for k in list(all_projected_boundary_pts.keys()):
        if all_projected_boundary_pts[k] is None:
            # remove index k from the list 
            print(f"Removing frame {k} with no projected boundary")
            del all_projected_boundary_pts[k]
        
        
    track_manager.add_matches(filtered_matches, kp2, int(frame_id))
# print("\nAll projected boundaries:")
# for frame_id, pts in all_projected_boundary_pts.items():
#     print(f"{frame_id}: {pts.reshape(4,2)}")    
    
    

[353, 359, 365, 400, 426, 449, 461, 471, 496, 531]

Processing frame 353
Keypoints: 124 vs 110
Good matches: 26
Inliers: 26
[[[1143.4841 1011.7938]]

 [[1196.2146 1028.2156]]

 [[1198.1584 1093.369 ]]

 [[1144.1079 1068.2979]]]

Processing frame 359
Keypoints: 124 vs 126
Good matches: 83
Inliers: 83
[[[1802.4426   999.0972 ]]

 [[1876.1599  1009.52704]]

 [[1871.443   1072.9948 ]]

 [[1796.7793  1054.6667 ]]]

Processing frame 365
Keypoints: 124 vs 117
Good matches: 69
Inliers: 69
[[[1485.1927 1170.8236]]

 [[1570.5729 1188.0221]]

 [[1568.4703 1267.3268]]

 [[1481.5669 1239.8792]]]

Processing frame 400
Keypoints: 124 vs 29
Good matches: 12
Inliers: 10
[[[1304.8461   835.6186 ]]

 [[1454.0413   851.8996 ]]

 [[1455.5815  1002.7271 ]]

 [[1307.4722   965.35956]]]

Processing frame 449
Keypoints: 124 vs 134
Good matches: 69
Inliers: 69
[[[1615.0599   907.15625]]

 [[1668.882    906.48956]]

 [[1666.9434   954.766  ]]

 [[1612.0955   950.2874 ]]]

Processing frame 461
Keypoints: 124 vs 1

In [57]:
import numpy as np
import json

"""

    Naming convention:
    ------------------
    T_ab : transforms a point from frame 'b' to frame 'a'

    So:
    - T_wc : camera → world  (given)
    - T_cw : world → camera  (needed for projection)

    A 3D point transforms as:
        X_w = T_wc @ X_c
        X_c = T_cw @ X_w

    This function computes:
        T_cw = inverse(T_wc)
"""

def load_camera_poses(json_path):
    with open(json_path, 'r') as f:
        poses_data = json.load(f)

    camera_poses = {}

    for frame_id, mat in poses_data.items():
        T = np.array(mat)  # 4x4

        # Extract R and t from T_cw
        R_cw = T[:3, :3]
        t_cw = T[:3, 3].reshape(3,1)

        # Convert to T_wc
        R_wc = R_cw.T
        t_wc = -R_wc @ t_cw
        
        # Extract R and t from T_cw
        # R_wc = T[:3, :3]
        # t_wc = T[:3, 3].reshape(3,1)

        camera_poses[int(frame_id)] = (R_wc, t_wc)

    return camera_poses

camera_poses = load_camera_poses("Data/poses.json")

In [60]:

def triangulate_points(all_projected_boundary_pts, camera_poses, K, keyframe_idx):
    points_3d = []

    frame_ids = list(all_projected_boundary_pts.keys())
    print("Frames with projected points:", frame_ids)
    # pick any 2 views for triangulation (you can extend later)
    for fi in frame_ids:
        if fi == keyframe_idx:
            continue
        f1 = keyframe_idx
        f2 = fi
        print(f"Triangulating using frames {f1} and {f2}")
    
        R1, t1 = camera_poses[int(f1)]
        R2, t2 = camera_poses[int(f2)]
        
        P1 = K @ np.hstack((R1, t1))
        P2 = K @ np.hstack((R2, t2))

        pts1 = all_projected_boundary_pts[f1]
        pts2 = all_projected_boundary_pts[f2]

        for i in range(4):
            pt1 = pts1[i].reshape(2,1)
            pt2 = pts2[i].reshape(2,1)

            point_4d = cv2.triangulatePoints(P1, P2, pt1, pt2)
            point_3d = (point_4d[:3] / point_4d[3]).reshape(3)

            points_3d.append(point_3d)

        
    points_3d = np.array(points_3d)
    
    return points_3d

def triangulate_tracks(tracks, camera_poses, K):
    points_3d = []

    for track_id, track in tracks.items():
        if len(track) < 2:
            continue

        # use first two observations
        (f1, pt1), (f2, pt2) = track[:2]

        R1, t1 = camera_poses[f1]
        R2, t2 = camera_poses[f2]

        P1 = K @ np.hstack((R1, t1))
        P2 = K @ np.hstack((R2, t2))

        pt1 = np.array(pt1).reshape(2,1)
        pt2 = np.array(pt2).reshape(2,1)

        X = cv2.triangulatePoints(P1, P2, pt1, pt2)
        X = (X[:3] / X[3]).ravel()

        if np.isfinite(X).all():
            points_3d.append(X)

    return np.array(points_3d)

In [61]:
print(K)
points_3d_boundary = triangulate_points(all_projected_boundary_pts, camera_poses, K, keyframe_idx)

print("3D points shape:", points_3d_boundary.shape)
print(points_3d_boundary)
print("Std dev:", np.std(points_3d_boundary, axis=0))
socket_pos = np.mean(points_3d_boundary, axis=0)
print("Socket position:", socket_pos)
min_vals = points_3d_boundary.min(axis=0)
max_vals = points_3d_boundary.max(axis=0)
extent = (max_vals - min_vals)
print("Min:", min_vals)
print("Max:", max_vals)
print("Extent:", extent)
local_pts = (points_3d_boundary - socket_pos)

[[1.47700975e+03 0.00000000e+00 1.29825015e+03]
 [0.00000000e+00 1.48044245e+03 6.86820162e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Frames with projected points: [426, 353, 359, 365, 400, 449, 461, 471, 531]
Triangulating using frames 426 and 353
Triangulating using frames 426 and 359
Triangulating using frames 426 and 365
Triangulating using frames 426 and 400
Triangulating using frames 426 and 449
Triangulating using frames 426 and 461
Triangulating using frames 426 and 471
Triangulating using frames 426 and 531
3D points shape: (32, 3)
[[0.27343664 0.22616392 0.5354443 ]
 [0.30369112 0.22761038 0.53726804]
 [0.3045052  0.22722512 0.5076128 ]
 [0.27324536 0.22577667 0.50876147]
 [0.27335727 0.22654158 0.53527665]
 [0.30368575 0.22757554 0.53734535]
 [0.30449387 0.22714181 0.5077381 ]
 [0.2731589  0.22613719 0.5086118 ]
 [0.27507943 0.22019014 0.53759444]
 [0.30512735 0.22228545 0.5392174 ]
 [0.3065709  0.21947362 0.5107419 ]
 [0.27550662 0.21767859 0.5119534 ]
 [0.273190

In [62]:
def visualize_tracks(image_indices, tracks, get_image_path, bboxes=None, max_tracks=30):
    """
    Visualize tracks across frames.

    - Same color = same track
    - Lines show motion across frames
    - max_tracks limits clutter
    """

    # limit number of tracks for clarity
    track_ids = list(tracks.keys())[:max_tracks]

    # assign color per track
    colors = {
        tid: tuple(np.random.randint(0, 255, 3).tolist())
        for tid in track_ids
    }

    for frame_id in image_indices:
        img = cv2.imread(get_image_path(frame_id)).copy()

        # draw bbox if available
        if bboxes is not None:
            bbox = bboxes.get(str(frame_id))
            if bbox is not None:
                x, y, w, h = bbox
                cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)

        for tid in track_ids:
            if tid not in tracks:
                continue

            observations = tracks[tid]

            pts = []
            for (f_id, pt) in observations:
                if f_id <= frame_id:
                    pts.append((int(pt[0]), int(pt[1])))

            # draw trajectory
            for i in range(1, len(pts)):
                cv2.line(img, pts[i-1], pts[i], colors[tid], 2)

            # draw current point
            for (f_id, pt) in observations:
                if f_id == frame_id:
                    x, y = int(pt[0]), int(pt[1])
                    cv2.circle(img, (x, y), 5, colors[tid], -1)

        cv2.imshow("Track Visualization", img)

        key = cv2.waitKey(0)
        if key == 27:  # ESC to exit early
            break

    cv2.destroyAllWindows()

In [63]:
valid_tracks = track_manager.get_valid_tracks()
points_3d_plane = triangulate_tracks(valid_tracks, camera_poses, K)
print("3D points from tracks :", points_3d_plane)

visualize_tracks(
    frame_ids,
    valid_tracks,
    get_image_path,
    bboxes=bboxes,
    max_tracks=70   # reduce clutter
)

3D points from tracks : [[0.26544985 0.22812254 0.7195443 ]
 [0.27184705 0.22770169 0.68642771]
 [0.27241176 0.22660318 0.68604198]
 [0.27175782 0.23727739 0.68007872]
 [0.28298731 0.22796954 0.52700082]
 [0.2841567  0.22769538 0.52108357]
 [0.27753989 0.22779305 0.68440687]
 [0.27738635 0.2284853  0.68667546]
 [0.27712745 0.22849028 0.72975895]
 [0.28813968 0.22500824 0.52374556]
 [0.28193877 0.22814508 0.68575717]
 [0.28191936 0.22857919 0.68844816]
 [0.28949801 0.22761047 0.52384941]
 [0.28965761 0.22705166 0.52411882]
 [0.28203886 0.22780493 0.70204018]
 [0.28284636 0.22633904 0.72581134]
 [0.29058514 0.2238097  0.56417188]
 [0.28474541 0.22881738 0.68630079]
 [0.29578709 0.22629026 0.52181359]
 [0.29661457 0.22905167 0.52690795]
 [0.29080026 0.22875056 0.70218126]
 [0.29083056 0.22884865 0.70214947]
 [0.29287895 0.23157989 0.70001268]
 [0.29643237 0.23022458 0.68060639]
 [0.29700307 0.22960204 0.6829995 ]
 [0.29832403 0.2292322  0.70670054]
 [0.30090279 0.22943716 0.68175751]
 [0.

In [64]:
import numpy as np
np.random.seed(42)

def fit_plane(p1, p2, p3):
    v1 = p2 - p1
    v2 = p3 - p1
    normal = np.cross(v1, v2)
    normal /= np.linalg.norm(normal)
    d = -np.dot(normal, p1)
    return normal, d

def point_plane_distance(points, normal, d):
    return np.abs(points @ normal + d)

points_3d = np.unique(points_3d_plane, axis=0)
best_inliers = []
best_model = None

for _ in range(500):
    ids = np.random.choice(len(points_3d_plane), 3, replace=False)
    p1, p2, p3 = points_3d_plane[ids]

    normal, d = fit_plane(p1, p2, p3)

    dist = point_plane_distance(points_3d_plane, normal, d)
    
    threshold = 0.002   # or tuned value
    inliers = points_3d_plane[dist < threshold]

    if len(inliers) > len(best_inliers):
        best_inliers = inliers
        best_model = (normal, d)
        
points_filtered = np.array(best_inliers)

centroid = points_filtered.mean(axis=0)
X = points_filtered - centroid

_, _, Vt = np.linalg.svd(X)

x_axis = Vt[0]
y_axis = Vt[1]
z_axis = Vt[2]

z_axis = 1 * Vt[2] / np.linalg.norm(Vt[2])
if z_axis[1] < 0:
    z_axis = -z_axis
    
x_axis = Vt[0] / np.linalg.norm(Vt[0])

y_axis = np.cross(x_axis, z_axis)
y_axis /= np.linalg.norm(y_axis)
if y_axis[0] < 0:
    y_axis = -y_axis
# recompute x to fix drift
x_axis = np.cross(y_axis, z_axis)
x_axis /= np.linalg.norm(x_axis)

if x_axis[2] < 0:
    x_axis = -x_axis
    
Rotation_matrix = np.column_stack((x_axis, y_axis, z_axis))
print("x_axis:", x_axis)
print("y_axis:", y_axis)
print("z_axis:", z_axis)
print("Rotation R:\n", Rotation_matrix)
print("Rotation R:\n", Rotation_matrix[2][0])

# --- Transform to object frame ---
local_pts_here = local_pts
local_pts_here = local_pts_here @ Rotation_matrix
# # --- Robust extent ---
min_vals = local_pts_here.min(axis=0)
max_vals = local_pts_here.max(axis=0)
extent = max_vals - min_vals

print("Extent:", extent)

x_axis: [0.16406228 0.01306025 0.98636352]
y_axis: [ 0.98525645  0.04700332 -0.1645005 ]
z_axis: [-0.04851078  0.99880935 -0.00515622]
Rotation R:
 [[ 0.16406228  0.98525645 -0.04851078]
 [ 0.01306025  0.04700332  0.99880935]
 [ 0.98636352 -0.1645005  -0.00515622]]
Rotation R:
 0.9863635229059736
Extent: [0.03713054 0.03674112 0.01275414]


/tmp/ipykernel_3752837/2443298953.py:8: RuntimeWarning: invalid value encountered in divide
  normal /= np.linalg.norm(normal)


In [65]:
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

# %matplotlib qt
# %matplotlib tk

def get_visible_face(corners_world, R_wc, t_wc):
    faces = [
        [0,1,2,3],
        [4,5,6,7],
        [0,1,5,4],
        [2,3,7,6],
        [1,2,6,5],
        [0,3,7,4]
    ]

    best_face = None
    best_score = -np.inf

    for face in faces:
        pts = corners_world[face]

        # Compute face normal (in world frame)
        v1 = pts[1] - pts[0]
        v2 = pts[2] - pts[0]
        normal = np.cross(v1, v2)
        normal = normal / np.linalg.norm(normal)
        normal = normal.flatten()

        center = np.mean(pts, axis=0).flatten()

        cam_center = (-R_wc.T @ t_wc).flatten()

        view_dir = center - cam_center
        view_dir = view_dir / np.linalg.norm(view_dir)
        view_dir = view_dir.flatten()

        score = float(np.dot(normal, view_dir))  # <-- FORCE scalar

        if score > best_score:
            best_score = score
            best_face = face

    return best_face

def load_obb(json_path, entity_name):
    with open(json_path, "r") as f:
        data = json.load(f)

    for obj in data:
        if obj["entity"] == entity_name:
            obb = obj["obb"]
            center = np.array(obb["center"])
            extent = np.array(obb["extent"])
            rotation = np.array(obb["rotation"])
            return center, extent, rotation

    raise ValueError(f"{entity_name} not found")

def get_obb_corners(center, extent, R):
    ex, ey, ez = extent / 2

    corners_local = np.array([
        [-ex, -ey, -ez],
        [ ex, -ey, -ez],
        [ ex,  ey, -ez],
        [-ex,  ey, -ez],
        [-ex, -ey,  ez],
        [ ex, -ey,  ez],
        [ ex,  ey,  ez],
        [-ex,  ey,  ez],
    ])

    return center + corners_local @ R.T

def project_points(points, K, R_wc, t_wc):
    projected = []
    valid_mask = []

    for p in points:
        p_cam = (R_wc @ p.reshape(3,1) + t_wc).flatten()
        
        if p_cam[2] <= 0:  # behind camera
            projected.append([np.nan, np.nan])
            valid_mask.append(False)
            continue

        p_img = K @ p_cam
        projected.append(p_img[:2] / p_img[2])
        valid_mask.append(True)

    return np.array(projected), np.array(valid_mask)

def get_front_face(points_world, R_wc, t_wc):
    depths = np.array([(R_wc @ p + t_wc)[2] for p in points_world])
    return np.argsort(depths)[:4]

def get_valid_polygon(corners_2d, valid_mask):
    pts = corners_2d[valid_mask]

    # remove NaNs
    pts = pts[~np.isnan(pts).any(axis=1)]

    if len(pts) < 4:
        return None

    return pts

def compute_iou(poly1, poly2):
    if poly1 is None or poly2 is None:
        return 0.0

    if len(poly1) < 4 or len(poly2) < 4:
        return 0.0

    try:
        p1 = Polygon(poly1).convex_hull
        p2 = Polygon(poly2).convex_hull

        if not p1.is_valid or not p2.is_valid:
            return 0.0

        inter = p1.intersection(p2).area
        union = p1.union(p2).area

        return inter / union if union > 0 else 0

    except:
        return 0.0

def draw_polygon(img, pts, color):
    pts = pts.astype(int)
    for i in range(len(pts)):
        cv2.line(img, tuple(pts[i]), tuple(pts[(i+1)%len(pts)]), color, 2)
        
def compare_obbs(
    pred_center, pred_extent, pred_R,
    gt_center, gt_extent, gt_R,
    image, K, R_wc, t_wc
):
    # --- Corners ---
    pred_corners = get_obb_corners(pred_center, pred_extent, pred_R)
    gt_corners   = get_obb_corners(gt_center, gt_extent, gt_R)

    # --- Projection ---
    pred_2d, pred_mask = project_points(pred_corners, K, R_wc, t_wc)
    gt_2d, gt_mask     = project_points(gt_corners, K, R_wc, t_wc)

    # --- Front face ---
   # --- Select visible face ---
    pred_face_idx = get_visible_face(pred_corners, R_wc, t_wc)
    gt_face_idx   = get_visible_face(gt_corners, R_wc, t_wc)

    # --- Extract polygons ---
    pred_poly = pred_2d[pred_face_idx]
    gt_poly   = gt_2d[gt_face_idx]
    
    # --- IoU ---
    iou = compute_iou(pred_poly, gt_poly)

    # --- Visualization ---
    img_vis = image.copy()
    draw_polygon(img_vis, pred_poly, (0,255,0))  # green = prediction
    draw_polygon(img_vis, gt_poly,   (0,0,255))  # red = ground truth

    return img_vis, iou

gt_center, gt_extent, gt_R = load_obb("Data/sample_answers.json", "vga_socket")


pred_center = socket_pos
pred_extent = extent
pred_R = Rotation_matrix
# pred_R = np.array([[-0.03877377, 0.99826172, -0.04438621],
#           [ 0.00967822, -0.0447927,   0.99894942],
#           [ 0.99920114, -0.03830346, -0.01139818]])

print("Predicted center:", pred_center)
print("Predicted extent:", pred_extent)
print("Predicted rotation:", pred_R)
R_wc, t_wc = camera_poses[426]

img = cv2.imread("Data/frame_000426.png")

img_vis, iou = compare_obbs(
    pred_center, pred_extent, pred_R,
    gt_center, gt_extent, gt_R,
    img, K, R_wc, t_wc
)

print("IoU:", iou)

# import matplotlib.pyplot as plt
# plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
# plt.title(f"IoU: {iou:.3f}")
# plt.show()

import matplotlib.pyplot as plt
import cv2

cv2.imshow("OBB Projection", img_vis)
cv2.waitKey(0)
cv2.destroyAllWindows()

Predicted center: [0.28875467 0.22675034 0.5223537 ]
Predicted extent: [0.03713054 0.03674112 0.01275414]
Predicted rotation: [[ 0.16406228  0.98525645 -0.04851078]
 [ 0.01306025  0.04700332  0.99880935]
 [ 0.98636352 -0.1645005  -0.00515622]]
IoU: 0.0


In [ ]:
# Code for writing the OBB to JSON

import json
import numpy as np

def update_entity_obb(json_path, entity_name, center, extent, rotation):
    # Convert to Python lists (important for JSON)
    center = np.asarray(center).tolist()
    extent = np.asarray(extent).tolist()
    rotation = np.asarray(rotation).tolist()

    # Load JSON
    with open(json_path, "r") as f:
        data = json.load(f)

    # Find and update entity
    found = False
    for obj in data:
        if obj["entity"] == entity_name:
            obj["obb"]["center"] = center
            obj["obb"]["extent"] = extent
            obj["obb"]["rotation"] = rotation
            found = True
            break

    if not found:
        raise ValueError(f"Entity '{entity_name}' not found in JSON")

    # Save back
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Updated {entity_name} successfully!")

file_dir = "Data"
file_name = "sample_answers.json"
file_path = os.path.join(file_dir, file_name)
# take care of rotation while dumping
# update_entity_obb(
#     file_path,
#     device_type,
#     socket_pos,
#     extent,
#     Rotation_matrix
# )